# EpiPiaui Monitor - fluxo do protótipo

Este notebook demonstra o fluxo mínimo: carregar notícias reais de 2024, gravar no SQLite, processar com PLN e consultar as menções extraídas.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))

from epipiaui_monitor.coletores import ColetorNoticiasSemeadas
from epipiaui_monitor.banco import inicializar_banco, salvar_noticias, carregar_noticias, limpar_mencoes, substituir_mencoes, carregar_mencoes
from epipiaui_monitor.configuracao import CAMINHO_BANCO_PADRAO
from epipiaui_monitor.pln import EpiPiauiPLN


## 1. Coleta das sementes reais

As sementes reais registram URLs localizadas em fontes piauienses entre janeiro e julho de 2024. Quando o HTML está disponível, o coletor baixa a matéria; quando o site bloqueia a coleta automatizada, usa texto curto de apoio marcado como reserva.

In [ ]:
inicializar_banco(CAMINHO_BANCO_PADRAO)
noticias = ColetorNoticiasSemeadas().coletar()
salvar_noticias(noticias, CAMINHO_BANCO_PADRAO)
carregar_noticias(CAMINHO_BANCO_PADRAO)[['fonte', 'titulo', 'data_publicacao', 'url']]


## 2. Processamento com PLN

O processamento detecta doenças, municípios e sintomas. A associação principal usa coocorrência por sentença.

In [ ]:
processador = EpiPiauiPLN()
mencoes = processador.processar_noticias(noticias)
limpar_mencoes(CAMINHO_BANCO_PADRAO)
substituir_mencoes(mencoes, CAMINHO_BANCO_PADRAO)
carregar_mencoes(CAMINHO_BANCO_PADRAO)[['doenca', 'municipio', 'sintomas', 'confianca', 'sentenca']]
